<img src="logo.png" alt="Vegeta" width="240">

# A boat on the sea — an unmanned survey vessel, from hull lines to life at sea

A 1 m electric unmanned surface vessel (USV) for coastal survey: hard-chine V-bottom hull, deck,
a transom bracket carrying the motor pod. The notebook covers what a small boat has to answer:

```
CAD (Dedalus): hull lines, shell, deck, bracket ─► mass budget, centre of gravity
hydrostatics from the mesh (numpy): draft, displacement, waterplane, wetted surface, GZ curve, GM
resistance: ITTC friction + form factor + wave allowance, double-body RANS (Aeromant) as a check
propulsion (Boreas in water): marine propeller + motor + battery ─► top speed, cruise power, endurance
structure (Talos): hull bottom under hydrostatic and slamming pressure, bracket under thrust and wave slap, bracket modes
three sea states (Chronos): wave encounter cycles ─► spectra ─► fatigue of hull and bracket ─► life ─► print the bracket
```

Every number is an explicit, coarse input (sea states, slamming factor, S-N curves). The value is in
the comparison and the record; a tank test and a sea trial validate the absolutes.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cadquery as cq
import pyvista as pv
from tqdm.auto import tqdm
from vegeta import dedalus, talos, aeromant, boreas, chronos, mellonia
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM

RUNS = Path("_runs/boat"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "survey_boat.py"
shutil.copy(Path("designs/survey_boat.py"), design_file)
boat_design = dedalus.load_design(f"{design_file}:SurveyBoat")
RHO_W, NU_W, G = 1025.0, 1.05e-6, 9.81          # sea water
pd.DataFrame(boat_design.params.table()).set_index("name")

## 1. Hull lines and parts

In [ ]:
p = boat_design.resolve()
parts_cad = {k: boat_design.generate(part=k) for k in ("boat", "hull_solid", "hull_shell", "bracket")}
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.1) for k, g in parts_cad.items()}
boat, hull_solid, hull_shell, bracket = (parts_cad[k] for k in ("boat", "hull_solid", "hull_shell", "bracket"))
dviz.show(dviz.plot3d(boat))

In [ ]:
fig = dviz.plot_sections(hull_shell, normal="x", positions=[80.0, 400.0, 750.0], cols=3)     # body plan: stations
fig = dviz.plot_sections(boat, normal="y", positions=[0.0], cols=1)                            # profile through the centreline

In [ ]:
RHO_PETG = 1.27e-3          # g/mm^3
items = pd.DataFrame([
    ("hull shell + deck (PETG, from CAD)", (boat.volume - bracket.volume) * RHO_PETG, 0.4 * p["length"], 70.0),
    ("transom bracket (PETG-CF, from CAD)", bracket.volume * 1.25e-3, -10.0, 60.0),
    ("motor pod + propeller", 300.0, -60.0, -25.0),
    ("battery 4S 10 Ah", 900.0, 380.0, 25.0),
    ("electronics, GPS, radio", 300.0, 500.0, 120.0),
    ("survey payload (sonar)", 500.0, 350.0, 15.0),
    ("hatches, fasteners, cabling", 250.0, 450.0, 100.0),
], columns=["item", "mass_g", "x_mm", "z_mm"]).set_index("item")
M_KG = items["mass_g"].sum() / 1000
CG = np.array([(items["mass_g"] * items["x_mm"]).sum() / items["mass_g"].sum(), 0.0, (items["mass_g"] * items["z_mm"]).sum() / items["mass_g"].sum()])
print(f"displacement {M_KG:.2f} kg | CG x {CG[0]:.0f} mm from the transom, z {CG[2]:.0f} mm above the keel")
items.round(1)

## 2. Hydrostatics and stability from the mesh

The outer hull is tessellated and clipped below a horizontal plane: the immersed volume is the
displacement, its centroid the centre of buoyancy. The draft is found by bisection on the mass; the
righting arm GZ at a heel angle is the horizontal distance between the centre of buoyancy of the heeled
hull and the centre of gravity (both in the earth frame). The deck-edge immersion angle is where the
closed CAD solid stops representing the boat (a real deck would flood).

In [ ]:
hull_mesh = dviz.to_pyvista(hull_solid, 0.3).triangulate().clean()

def immersed(draft_mm, heel_deg=0.0):
    m = hull_mesh.rotate_x(heel_deg, point=(0, 0, 0), inplace=False) if heel_deg else hull_mesh
    cl = m.clip_closed_surface(normal=(0, 0, -1), origin=(0, 0, draft_mm))
    return (cl.volume * 1e-9 * RHO_W, np.array(cl.center_of_mass()), cl) if cl.n_points else (0.0, np.zeros(3), cl)

def draft_for(mass_kg, heel_deg=0.0):
    lo, hi = 0.0, p["depth"]
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if immersed(mid, heel_deg)[0] < mass_kg else (lo, mid)
    return hi

T = draft_for(M_KG)
disp, CB, clipped = immersed(T)
wl = hull_mesh.slice(normal=(0, 0, 1), origin=(0, 0, T))
wp_pts = wl.points[:, :2]
def polygon_area(pts):                       # shoelace on the convex-ish waterplane outline
    c = pts.mean(axis=0); ang = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0]); q = pts[np.argsort(ang)]
    return 0.5 * abs(np.dot(q[:, 0], np.roll(q[:, 1], 1)) - np.dot(q[:, 1], np.roll(q[:, 0], 1)))
A_wp = polygon_area(wp_pts)
S_wet = (clipped.area - A_wp) * 1e-6                                   # m^2 (clip cap removed)
I_wp = polygon_area(wp_pts) * 0 + np.sum((wp_pts[:, 1] - wp_pts[:, 1].mean()) ** 2) / len(wp_pts) * A_wp   # rough I_T = A * <y^2>
BM = I_wp / (disp / RHO_W * 1e9); KB = CB[2]; GM_est = KB + BM - CG[2]
freeboard = p["depth"] - T
hydro = pd.Series({"draft_mm": T, "displacement_kg": disp, "freeboard_mm": freeboard, "LCB_mm": CB[0], "LCG_mm": CG[0],
                   "trim_note": "bow-down" if CG[0] > CB[0] + 5 else ("stern-down" if CG[0] < CB[0] - 5 else "level"),
                   "waterplane_area_m2": A_wp * 1e-6, "wetted_surface_m2": S_wet, "KB_mm": KB, "BM_mm (rough)": BM, "GM_mm (KB+BM-KG)": GM_est})
hydro.map(lambda v: round(v, 2) if isinstance(v, float) else v)

In [ ]:
heels = np.arange(0, 61, 5)
def righting_arm(h):
    Th = draft_for(M_KG, h)
    dh, cbh, _ = immersed(Th, h)
    r = math.radians(h)
    cg_r = np.array([CG[0], CG[1] * math.cos(r) - CG[2] * math.sin(r), CG[1] * math.sin(r) + CG[2] * math.cos(r)])
    deck_edge_z = min(-p["beam"] / 2 * math.sin(r), p["beam"] / 2 * math.sin(r)) + p["depth"] * math.cos(r)   # lower deck edge, earth frame
    return cg_r[1] - cbh[1], deck_edge_z < Th                        # righting arm (+ = restoring), deck edge immersed?
gz, dfl = [], None
for h in heels:
    a, wet = righting_arm(h)
    b, _ = righting_arm(-h)                                           # average both sides: cancels tessellation asymmetry
    gz.append(0.5 * (a - b))
    if dfl is None and wet:
        dfl = h
gz = np.array(gz)
GM = (gz[1] - gz[0]) / math.radians(heels[1] - heels[0])             # small-angle slope
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(heels, gz, "o-"); ax[0].axhline(0, color="k", lw=0.8)
ax[0].plot([0, 30], [0, GM * math.radians(30)], "--", color="#c62828", label=f"GM slope: {GM:.0f} mm")
if dfl: ax[0].axvline(dfl, color="#888", ls=":", label=f"deck edge immerses ~{dfl}°")
ax[0].set(xlabel="heel [deg]", ylabel="GZ [mm]", title="righting arm curve"); ax[0].legend(); ax[0].grid(alpha=0.3)
sec = dviz.section_polylines(hull_solid, "x", 400.0, 0.3)
for poly in sec: ax[1].plot(poly[:, 0], poly[:, 1], color="#1f3b5a")
ax[1].axhline(T, color="#1976d2", label=f"waterline, draft {T:.0f} mm"); ax[1].scatter([CG[1]], [CG[2]], color="#c62828", label="CG"); ax[1].scatter([CB[1]], [CB[2]], color="#2e7d32", label="CB")
ax[1].set_aspect("equal"); ax[1].set(xlabel="y [mm]", ylabel="z [mm]", title="midship section"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout()
print(f"GM from the GZ slope {GM:.0f} mm (formula estimate {GM_est:.0f} mm); max GZ {gz.max():.0f} mm at {heels[gz.argmax()]}°")

## 3. Resistance and propulsion

Design estimate: ITTC-57 friction on the wetted surface × form factor (1 + k = 1.25, assumed for a
chine hull) + a wave-resistance allowance rising with Froude number (assumed hump curve). The
double-body RANS below is a *check on the viscous part*: the immersed hull mirrored at the waterline in
single-phase water. Its flat transom (dry on the real boat at speed) and the coarse wall mesh make it an
upper bound — read it as such.

In [ ]:
L_WL = p["length"] / 1000
FORM_FACTOR = 1.25
def resistance(V):
    Re = V * L_WL / NU_W
    Cf = 0.075 / (math.log10(Re) - 2) ** 2
    Fn = V / math.sqrt(G * L_WL)
    Cw = 0.004 * (Fn / 0.45) ** 4 / (1 + (Fn / 0.45) ** 4) * 1.6          # assumed wave-resistance hump for a small chine hull
    q = 0.5 * RHO_W * V ** 2
    return q * S_wet * (FORM_FACTOR * Cf + Cw), Fn, Cf, Cw

Vs = np.linspace(0.3, 5.0, 48)
R = np.array([resistance(v)[0] for v in Vs])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(Vs, R, label="total (ITTC x 1.25 + wave allowance)"); ax[0].plot(Vs, [0.5 * RHO_W * v**2 * S_wet * resistance(v)[2] for v in Vs], "--", label="friction only")
ax[0].set(xlabel="speed [m/s]", ylabel="resistance [N]"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot([resistance(v)[1] for v in Vs], R); ax[1].set(xlabel="Froude number", ylabel="resistance [N]"); ax[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
V_CFD = 2.0
under = cq.Workplane("XY").add(hull_solid.shape).intersect(cq.Workplane("XY").box(2 * p["length"], 2 * p["beam"], T, centered=(True, True, False)))
double_body = dedalus.Geometry.from_cadquery(under.union(under.mirror("XY", basePointVector=(0, 0, T))), name="double_body")
db = double_body.export(RUNS / "cad" / "double_body", formats=("stl",), stl_tolerance=0.3)
case = aeromant.CFDCase("rans_ksst_external", db.artifacts["stl"],
    dict(velocity=V_CFD, kinematic_viscosity=NU_W, density=RHO_W, reference_area=2 * S_wet, reference_length=0.25,
         center_of_rotation=(0.5, 0, T / 1000), iterations=250, residual_target=1e-4, surface_level=4, near_level=3, wake_level=2),
    workdir=RUNS / "cfd_double_body", geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
case.prepare(overwrite=True)
cfd = case.run(progress=True)
R_est, Fn, Cf, Cw = resistance(V_CFD)
if cfd.ok:
    R_cfd = cfd.metrics["drag_force_N"] / 2
    print(f"at {V_CFD} m/s: double-body RANS viscous+form drag {R_cfd:.1f} N (upper bound: transom base drag, coarse wall mesh) | "
          f"ITTC friction {0.5 * RHO_W * V_CFD**2 * S_wet * Cf:.1f} N | design estimate incl. waves {R_est:.1f} N | converged: {cfd.metrics['converged']}")
else:
    print("CFD not available:", cfd.messages[:1])

In [ ]:
if cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="y", origin=(0, 0.0, 0)))
    fig = aviz.plot_section(case, "p", normal="z", origin=(0, 0, T / 1000 * 0.5), zoom=2)   # pressure on a plane inside the immersed hull depth

In [ ]:
prop = boreas.Propeller.from_pitch("60 mm 3-blade marine", 0.060, 0.050, blades=3, chord_root_m=0.012, chord_max_m=0.018,
                                   chord_tip_m=0.008, mass_kg=0.02, rotor_mass_kg=0.05, notes="generic planform; fit to the real propeller")
section = boreas.Airfoil(name="marine blade section", cl_alpha=5.5, alpha0_deg=-2.0, cl_max=1.0, cd0=0.02, k=0.05, source="assumed")
motor = boreas.Motor("2836-500KV (water-cooled pod)", kv_rpm_per_volt=500, resistance_ohm=0.08, no_load_current_a=0.8, max_current_a=30, mass_kg=0.12)
battery = boreas.Battery("4S 10 Ah", cells=4, capacity_ah=10.0, usable_fraction=0.8, mass_kg=0.9)
drive = boreas.Propulsion(prop, section, motor, battery, rho=RHO_W)
WAKE = 0.9                                                             # water reaches the propeller at 0.9 x boat speed (assumed wake fraction)
T_full = np.array([drive.at_throttle(1.0, WAKE * v).thrust for v in Vs])
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(Vs, R, color="#c62828", label="resistance (design estimate)"); ax.plot(Vs, T_full, label="thrust at full throttle")
ax.set(xlabel="speed [m/s]", ylabel="N", title="thrust available vs resistance"); ax.legend(); ax.grid(alpha=0.3)
ok = np.where(T_full > R)[0]
V_MAX = Vs[ok.max()] if len(ok) else float("nan")
if len(ok) and ok.max() == len(Vs) - 1:
    print("thrust exceeds the resistance estimate over the whole range: the resistance model (planing regime) is the limit here, not the drive")
V_CRUISE = 1.5
cruise = drive.for_thrust(resistance(V_CRUISE)[0], WAKE * V_CRUISE)
full = drive.at_throttle(1.0, WAKE * V_MAX)
endurance_h = battery.usable_wh / cruise.electrical_power
print(f"top speed ~{V_MAX:.1f} m/s ({V_MAX * 1.944:.1f} kn) | cruise {V_CRUISE} m/s: {cruise.rpm:.0f} rpm, {cruise.electrical_power:.0f} W, "
      f"{endurance_h:.1f} h, {V_CRUISE * endurance_h * 3.6:.0f} km | full throttle {full.rpm:.0f} rpm, {full.current:.1f} A" + (" (CURRENT LIMITED)" if full.current_limited else ""))
pd.DataFrame({k: boreas.excitations(prop, v.rpm) for k, v in (("cruise", cruise), ("full", full))}).round(2)

## 4. Structure: hull bottom, transom bracket, and the bracket's modes

Unit cases (all linear, so any load level is a multiple):

| pattern | unit case | level unit |
|---|---|---|
| `bottom_kpa` | 1 kPa on the bottom panels below the chine, deck edge fixed | kPa |
| `thrust` | 10 N forward on the pod, bracket bolted at its four holes | N |
| `pod_side` | 10 N sideways on the pod (wave slap, steering) | N |

Static checks: hydrostatic pressure at the keel (ρ g T), a slamming pressure from a coarse Wagner-type
estimate p = ½ ρ v² k(β) with the relative impact velocity of the roughest sea state, full thrust, and a
wave slap of the pod.

In [ ]:
PETG = talos.Material("PETG (printed hull)", youngs_modulus=2000.0, poissons_ratio=0.38, density=1.27e-9, yield_strength=45.0, source="nominal, XY orientation")
PETG_CF = talos.Material("PETG-CF (bracket)", youngs_modulus=4800.0, poissons_ratio=0.38, density=1.25e-9, yield_strength=45.0, source="nominal")
info = talos.inspect_step(cad["hull_shell"].artifacts["step"], units="mm-N-MPa")
bottom_tags = [s.tag for s in info.surfaces if s.bbox_max[2] < p["depth"] * 0.45 and s.area > 500]        # bottom panels below the chine
HULL_REGIONS = [talos.SurfacesInBox("deck_edge", (-1, -p["beam"], p["depth"] - 8, p["length"] + 1, p["beam"], p["depth"] + 1)),
                talos.Surfaces("bottom", bottom_tags)]
hull_model = talos.StructuralModel(cad["hull_shell"].artifacts["step"], "mm-N-MPa", PETG, HULL_REGIONS, [talos.FixedSupport("deck_edge")],
                                   [talos.Pressure("bottom", 0.001)], talos.MeshSettings(element_size=8.0), name="bottom_kpa")
hull_mesh_res = hull_model.mesh(RUNS / "hull_bottom", progress=True)
res_hull = hull_model.solve(RUNS / "hull_bottom", progress=True)
print(res_hull)
tviz.show(tviz.plot_results(res_hull, field="von_mises"))

In [ ]:
t_b, w_b, h_b = p["bracket_thickness"], p["bracket_width"], p["bracket_height"]
BR_REGIONS = [talos.SurfacesInBox("bolts", (-t_b - 1, -w_b / 2 + 3, -h_b + 15, 1, w_b / 2 - 3, 35)),
              talos.SurfacesInBox("pod", (-t_b - 92, -p["pod_diameter"] / 2 - 1, -h_b - p["pod_diameter"] / 2 - 1, -t_b + 1, p["pod_diameter"] / 2 + 1, -h_b + p["pod_diameter"] / 2 + 1))]
def bracket_model(loads, name, masses=()):
    return talos.StructuralModel(cad["bracket"].artifacts["step"], "mm-N-MPa", PETG_CF, BR_REGIONS, [talos.FixedSupport("bolts")], loads,
                                 talos.MeshSettings(element_size=3.0), name=name, masses=list(masses))
bracket_model([], "mesh", masses=[talos.PointMass("pod", 300e-6)]).mesh(RUNS / "bracket_mesh")
def bracket_case(name):
    shutil.copytree(RUNS / "bracket_mesh", RUNS / f"bracket_{name}", dirs_exist_ok=True)
    return RUNS / f"bracket_{name}"
res_thrust = bracket_model([talos.Force("pod", fx=10.0)], "thrust").solve(bracket_case("thrust"))
res_side = bracket_model([talos.Force("pod", fy=10.0)], "pod_side").solve(bracket_case("side"))
modes = bracket_model([], "modal", masses=[talos.PointMass("pod", 300e-6)]).solve_modes(bracket_case("modal"), n_modes=6)
print("bracket modes with the 300 g pod:", [round(f, 1) for f in modes.metrics["frequencies_hz"]], "Hz")
tviz.show(tviz.plot_results(res_thrust, field="von_mises"))

In [ ]:
structure = chronos.Structure(tuple(modes.metrics["frequencies_hz"]), damping_ratio=0.03)
fig = structure.campbell({"1P": 1, "3P": prop.blades}, np.linspace(500, 6000, 25), operating_rpm={"cruise": cruise.rpm, "full": full.rpm})
tviz.show(tviz.plot_mode(modes, mode=1))

In [ ]:
# sea states used here and below (significant wave height, peak period) — engineer inputs
SEA = {"harbour": {"Hs": 0.05, "Tp": 2.0}, "coastal chop": {"Hs": 0.30, "Tp": 3.0}, "open sea": {"Hs": 0.80, "Tp": 5.0}}
def deadrise_factor(beta_deg):            # Wagner-type peak pressure factor for a wedge, coarse
    return (math.pi / (2 * math.tan(math.radians(beta_deg)))) ** 2 * 0.15
def slam_pressure_kpa(Hs, Tp, V):
    v_wave = math.pi * Hs / Tp                                     # wave particle / vertical velocity scale
    v_rel = v_wave + 0.15 * V                                       # plus a share of the forward speed on the bow deadrise
    return 0.5 * RHO_W * v_rel ** 2 * deadrise_factor(p["deadrise_deg"]) / 1000
p_hydro = RHO_W * G * T / 1000 / 1000                              # kPa at the keel
p_slam = slam_pressure_kpa(SEA["open sea"]["Hs"], SEA["open sea"]["Tp"], V_MAX)
unit = {"bottom_kpa": (res_hull, 1.0), "thrust": (res_thrust, 10.0), "pod_side": (res_side, 10.0)}
def stress(pattern, level):
    r, u = unit[pattern]; return r.metrics["max_von_mises"] * level / u
F_slap = 0.5 * RHO_W * (math.pi * (p["pod_diameter"] / 2000) ** 2 * 3 + 0.02) * (1.5 * math.pi * SEA["open sea"]["Hs"] / SEA["open sea"]["Tp"]) ** 2 * 2
static = pd.DataFrame({
    "hull, hydrostatic at the keel": {"level": f"{p_hydro:.2f} kPa", "max_von_mises_MPa": stress("bottom_kpa", p_hydro), "SF_yield": PETG.yield_strength / stress("bottom_kpa", p_hydro)},
    "hull, slamming (open sea, top speed)": {"level": f"{p_slam:.1f} kPa", "max_von_mises_MPa": stress("bottom_kpa", p_slam), "SF_yield": PETG.yield_strength / stress("bottom_kpa", p_slam)},
    "bracket, full thrust": {"level": f"{full.thrust:.1f} N", "max_von_mises_MPa": stress("thrust", full.thrust), "SF_yield": PETG_CF.yield_strength / stress("thrust", full.thrust)},
    "bracket, wave slap on the pod": {"level": f"{F_slap:.1f} N", "max_von_mises_MPa": stress("pod_side", F_slap), "SF_yield": PETG_CF.yield_strength / stress("pod_side", F_slap)},
}).T
static

## 5. Three sea states as missions (Chronos)

Head seas: the encounter frequency is `f_e = f_w (1 + 2π f_w V / g)`; every wave is one cycle of
bottom pressure (a share of the slamming pressure, plus the hydrostatic mean), one cycle of pod slap
and a thrust fluctuation. Full-throttle bursts and station-keeping are segments of their own. Levels are
kPa for the hull, N for the bracket.

In [ ]:
def sea_segment(name, duration_s, sea, V, drive_point, thrust_swing=0.3, repeat=1):
    Hs, Tp = SEA[sea]["Hs"], SEA[sea]["Tp"]
    f_w = 1.0 / Tp
    f_e = f_w * (1 + 2 * math.pi * f_w * V / G)
    p_wave = 0.35 * slam_pressure_kpa(Hs, Tp, V)                                  # typical wave, not the worst slam
    slap = 0.5 * RHO_W * (math.pi * (p["pod_diameter"] / 2000) ** 2 * 3 + 0.02) * (math.pi * Hs / Tp) ** 2 * 2
    exc = (chronos.Excitation(f"wave pressure {sea}", f_e, p_wave, "bottom_kpa"),
           chronos.Excitation(f"pod slap {sea}", f_e, slap, "pod_side"),
           chronos.Excitation(f"thrust swing {sea}", f_e, thrust_swing * drive_point.thrust, "thrust"))
    return chronos.Segment(name, duration_s, {"bottom_kpa": p_hydro, "thrust": drive_point.thrust}, exc, repeat=repeat)

missions = {
    "harbour survey": chronos.Mission("harbour survey", (
        sea_segment("survey lines", 3000, "harbour", V_CRUISE, cruise),
        chronos.Segment("station keeping", 600, {"bottom_kpa": p_hydro, "thrust": 0.5}),
    ), "60 min in flat water"),
    "coastal transect": chronos.Mission("coastal transect", (
        sea_segment("transit out", 900, "coastal chop", V_CRUISE, cruise),
        sea_segment("full-throttle burst", 20, "coastal chop", V_MAX, full, repeat=6),
        sea_segment("survey lines", 1500, "coastal chop", V_CRUISE, cruise),
        chronos.Segment("slam on a wake", 1, {"bottom_kpa": p_slam * 0.6, "thrust": cruise.thrust}, repeat=10),
    ), "45 min in 0.3 m chop with six sprints"),
    "open sea": chronos.Mission("open sea", (
        sea_segment("transit", 1200, "open sea", V_CRUISE, cruise, thrust_swing=0.5),
        sea_segment("heavy going", 15, "open sea", V_MAX, full, thrust_swing=0.6, repeat=8),
        chronos.Segment("hard slam", 1, {"bottom_kpa": p_slam, "thrust": full.thrust}, repeat=12),
        sea_segment("return", 1200, "open sea", V_CRUISE, cruise, thrust_swing=0.5),
    ), "40 min in 0.8 m seas with twelve hard slams"),
}
for m in missions.values():
    fig = m.profile()
spectra = {k: chronos.build_spectrum(m, structure) for k, m in missions.items()}
for k, sp in spectra.items():
    sp.save(RUNS / f"spectrum_{k.replace(' ', '_')}.json")
    fig = sp.plot()
pd.DataFrame({k: {"duration_min": m.duration_h * 60, "blocks": len(spectra[k].blocks), "wave_cycles": spectra[k].total_cycles} for k, m in missions.items()}).T.round(0)

## 6. Fatigue of hull and bracket, damage maps, life over a usage mix

S-N curves — **assumed**: PETG hull σ_f = 60 MPa, b = −0.12, ultimate 45; PETG-CF bracket σ_f = 80,
b = −0.11, ultimate 55. Printed parts in sea water and sun degrade; coupon tests, aged, are the real input.

In [ ]:
CURVE_HULL = talos.FatigueCurve("PETG (assumed)", sigma_f=60.0, b=-0.12, ultimate=45.0, source="assumed")
CURVE_BR = talos.FatigueCurve("PETG-CF (assumed)", sigma_f=80.0, b=-0.11, ultimate=55.0, source="assumed")
hull_unit = {"bottom_kpa": (res_hull, 1.0)}
br_unit = {"thrust": (res_thrust, 10.0), "pod_side": (res_side, 10.0)}
def only(spec, patterns):
    return chronos.LoadSpectrum(spec.mission, spec.duration_s, [b for b in spec.blocks if b.pattern in patterns], list(patterns)).to_dict()
fat_hull = {k: talos.assess_fatigue(hull_unit, only(sp, ["bottom_kpa"]), CURVE_HULL, workdir=RUNS / f"fatigue_hull_{k.replace(' ', '_')}") for k, sp in tqdm(spectra.items(), desc="hull")}
fat_br = {k: talos.assess_fatigue(br_unit, only(sp, ["thrust", "pod_side"]), CURVE_BR, workdir=RUNS / f"fatigue_bracket_{k.replace(' ', '_')}") for k, sp in tqdm(spectra.items(), desc="bracket")}
hours = {k: m.duration_h for k, m in missions.items()}
life = pd.DataFrame({k: {"mission_h": hours[k], "hull damage/mission": fat_hull[k].result.metrics["damage_per_pass"], "hull hours to failure": fat_hull[k].result.metrics["hours_to_failure"],
                         "bracket damage/mission": fat_br[k].result.metrics["damage_per_pass"], "bracket hours to failure": fat_br[k].result.metrics["hours_to_failure"]} for k in missions}).T
life

In [ ]:
worst = life["hull damage/mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fat_hull[worst], res_hull.artifacts["mesh"]))

In [ ]:
worst_b = life["bracket damage/mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fat_br[worst_b], res_thrust.artifacts["mesh"]))

In [ ]:
usage = {"harbour survey": 0.5, "coastal transect": 0.35, "open sea": 0.15}
def rate(fat):
    d = {k: f.result.metrics["damage_per_pass"] for k, f in fat.items()}
    return sum(usage[k] * d[k] for k in usage) / sum(usage[k] * hours[k] for k in usage), d
r_hull, d_hull = rate(fat_hull); r_br, d_br = rate(fat_br)
sim = chronos.simulate_life({k: d_hull[k] + d_br[k] for k in missions}, hours, usage, n_flights=20000, seed=0)   # combined, conservative
fig = sim.plot()
mixes = {"as above": usage, "open-sea-only": {"harbour survey": 0.0, "coastal transect": 0.0, "open sea": 1.0}, "harbour-only": {"harbour survey": 1.0, "coastal transect": 0.0, "open sea": 0.0}}
def per_1000h(d, mix):
    return sum(mix[k] * d[k] for k in mix) / sum(mix[k] * hours[k] for k in mix) * 1000
pd.DataFrame({n: {"hull damage per 1000 h": per_1000h(d_hull, mix), "bracket damage per 1000 h": per_1000h(d_br, mix)} for n, mix in mixes.items()}).T

## 7. Print the bracket (Mellonia)

Plate flat on the bed, pod bore vertical: the layers run along the plate, across the bending of the
thrust load — the orientation the material numbers assume.

In [ ]:
prn = mellonia.slice_stl(cad["bracket"].artifacts["stl"], GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_y=90), RUNS / "print_bracket")
print(prn)
if prn.ok:
    mviz.show(mviz.plot_toolpath(prn))

## 8. Export

In [ ]:
doc = {"design": {"file": "designs/survey_boat.py", "parameters": p}, "mass_kg": M_KG, "cg_mm": CG.tolist(),
       "hydrostatics": {k: (float(v) if not isinstance(v, str) else v) for k, v in hydro.items()}, "gz_curve": {"heel_deg": heels.tolist(), "gz_mm": gz.tolist(), "GM_mm": GM, "deck_edge_deg": dfl},
       "resistance_design_N": dict(zip([round(v, 2) for v in Vs], R.round(2).tolist())),
       "cfd_double_body": {"speed": V_CFD, "half_drag_N": cfd.metrics.get("drag_force_N", float("nan")) / 2 if cfd.ok else None, "converged": cfd.metrics.get("converged") if cfd.ok else None},
       "propulsion": {"top_speed_m_s": V_MAX, "cruise": cruise.to_dict(), "full": full.to_dict(), "endurance_h": endurance_h},
       "bracket_modes_hz": modes.metrics["frequencies_hz"], "static_checks": static.to_dict(),
       "sea_states": SEA, "missions": {k: m.describe() for k, m in missions.items()},
       "damage_per_mission": {"hull": d_hull, "bracket": d_br}, "hours_per_mission": hours, "usage": usage,
       "damage_per_1000h": {"hull": r_hull * 1000, "bracket": r_br * 1000}}
(RUNS / "boat.json").write_text(json.dumps(doc, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**What to do with this:** the GZ curve and freeboard say whether the boat comes back upright and stays
dry — the numbers to compare against the sea state you will actually operate in; the resistance and
thrust curves give the speed and endurance the mission plan can promise; the slamming case and the
open-sea damage column say whether the hull bottom or the bracket needs more material for the rough
days. Every assumption above (wake fraction, form factor, slamming factor, S-N curves, sea states) is a
named input — change it, re-run, and the record shows what moved.